# Chapter 7 · 지역·기간 강건성

**연구질문:** 지역·기간을 바꿔도 모형 순위가 유지되는가? 창 길이와 시차에 민감한가?

> 위에서 아래로 실행하면 끝까지 동작합니다.

## 1. 핵심 개념·수식

**누수(leakage):** 예측 시점 $t$에 알 수 없는 정보가 특성·전처리·분할에 들어가 성능이 과대평가되는 문제. 대표 경로 세 가지.
1. **전처리 누수:** 표준화·결측대체·모수 추정을 전체 자료로 한다(학습구간 통계만 써야 함).
2. **미래 정보 누수:** 중심 이동평균 등 $y_{t+k}$를 포함하는 특성.
3. **분할 누수:** 시계열에 무작위 K-겹 교차검증을 쓴다(인접 관측이 학습·검증에 섞임).

**Walk-forward(롤링 오리진) 평가:** 시점 $T_1<T_2<\dots$마다 $[1,T_k]$로 학습하고 $(T_k,T_k+h]$를 평가한다. 폴드별 오차의 분포(평균·분산·최악)를 본다.

**Diebold–Mariano 검정:** 손실차 $d_t=L(e^{A}_t)-L(e^{B}_t)$의 평균이 0인지 검정한다. $\mathrm{DM}=\bar d/\sqrt{\widehat{\mathrm{LRV}}(d)/n}$.

**부트스트랩 신뢰구간:** 폴드 손실차를 블록 부트스트랩하여 우열의 불확실성을 본다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def parse_fhfa_master(raw):
    # FHFA hpi_master.csv 파싱. 스키마(열 이름)는 버전에 따라 달라질 수 있어 방어적으로 처리하고, 실패하면 예외를 낸다.
    df = pd.read_csv(io.BytesIO(raw) if isinstance(raw, (bytes, bytearray)) else io.StringIO(raw))
    df.columns = [c.strip().lower() for c in df.columns]
    if not {"yr", "period"}.issubset(df.columns):
        raise ValueError("열(yr, period) 없음: " + ", ".join(df.columns[:12]))
    col = "index_nsa" if "index_nsa" in df.columns else None
    if col is None:
        raise ValueError("index_nsa 열 없음")
    if "frequency" in df.columns:
        df = df[df["frequency"].astype(str).str.lower().str.startswith("month")]
    keys = [k for k in ("hpi_type", "hpi_flavor", "level", "place_id", "place_name") if k in df.columns]
    g = df.groupby(keys).size().sort_values(ascending=False)
    if keys:
        df = df.merge(g.index[:1].to_frame(index=False), on=keys)   # 관측이 가장 많은 한 계열
    s = pd.Series(pd.to_numeric(df[col], errors="coerce").values,
                  index=pd.to_datetime(dict(year=df["yr"].astype(int), month=df["period"].astype(int), day=1)))
    s = s.dropna().sort_index()
    if len(s) < 100:
        raise ValueError("관측 수 부족")
    return s

def synthetic_house(n=300, seed=SEED):
    rng = np.random.default_rng(seed); t = np.arange(n)
    cycle = 6 * np.sin(2 * np.pi * t / 120)
    seas = 1.2 * np.sin(2 * np.pi * (t % 12) / 12 - 1.0)
    shock = np.cumsum(rng.normal(0, 0.35, n))
    return pd.Series(100 + 0.35 * t + cycle + seas + shock, index=pd.date_range("2000-01-01", periods=n, freq="MS"))

def synthetic_volume(n=240, seed=SEED + 1):
    rng = np.random.default_rng(seed); t = np.arange(n)
    seas = 18 * np.sin(2 * np.pi * (t % 12) / 12 - 1.5)
    return pd.Series(np.maximum(200 + 0.2 * t + seas + np.cumsum(rng.normal(0, 1.2, n)) + rng.normal(0, 6, n), 10),
                     index=pd.date_range("2005-01-01", periods=n, freq="MS"))

def load_realestate(which="price"):
    if which == "volume":
        return dict(y=synthetic_volume().rename("volume"), m=12, freq="MS", label="월별 주택거래량(합성)", unit="건",
                    kind="SYNTHETIC (NOT real data): 검증된 무료 거래량 출처 없음")
    s, kind = None, "SYNTHETIC (NOT real data): 추세+주기+계절+확률적 충격 주택가격지수"
    if USE_LIVE_DATA:
        try:
            s = parse_fhfa_master(fetch_bytes("https://www.fhfa.gov/hpi/download/monthly/hpi_master.csv", timeout=90))
            kind = "REAL: FHFA HPI (index_nsa, 계열 선택 규칙은 parse_fhfa_master 참고)"
        except Exception as e:
            print("FHFA 수집/파싱 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if s is None:
        s = synthetic_house()
    return dict(y=s.rename("hpi"), m=12, freq="MS", label="주택가격지수", unit="지수", kind=kind)

D = load_realestate("price")
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 지역·기간 강건성

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_X(s, lags=(1, 2, 3, 7, 14), rolls=(7, 28)):
    X = pd.DataFrame({f"lag_{k}": s.shift(k) for k in lags})
    for w in rolls:
        X[f"rm_{w}"] = s.shift(1).rolling(w).mean()
    return X

def rolling_origin(s, n_folds=8, fold=None, kind="ridge", window=None, lags=(1, 2, 3, 7, 14)):
    s = s.astype(float); X = make_X(s, lags=lags); data = pd.concat([X, s.rename("target")], axis=1).dropna()
    XX, yy = data.drop(columns="target"), data["target"]; n = len(data); fold = fold or max(30, n // (n_folds + 3))
    start = n - n_folds * fold; out = []
    for k in range(n_folds):
        a, b = start + k * fold, start + (k + 1) * fold
        lo = 0 if window is None else max(0, a - window)
        mdl = make_pipeline(StandardScaler(), Ridge(alpha=1.0)) if kind == "ridge" else HistGradientBoostingRegressor(max_iter=120, learning_rate=0.06, random_state=SEED)
        mdl.fit(XX.iloc[lo:a], yy.iloc[lo:a]); p = mdl.predict(XX.iloc[a:b]); naive = XX["lag_1"].iloc[a:b].to_numpy()
        out.append({"fold": k + 1, "end": XX.index[b - 1], "RMSE": rmse(yy.iloc[a:b], p), "RMSE_naive": rmse(yy.iloc[a:b], naive)})
    return pd.DataFrame(out)

In [ ]:
# ---- 지역·기간 강건성: 여러 지역(합성 패널) × 시기 ----
rng = np.random.default_rng(SEED)
def synth_region(seed, trend, amp, vol, n=240):
    rg = np.random.default_rng(seed); t = np.arange(n)
    return pd.Series(100 + trend * t + amp * np.sin(2 * np.pi * (t % 12) / 12) + np.cumsum(rg.normal(0, vol, n)),
                     index=pd.date_range("2005-01-01", periods=n, freq="MS"))
regions = {"지역A(완만)": synth_region(1, 0.25, 1.0, 0.3), "지역B(변동 큼)": synth_region(2, 0.40, 2.5, 0.9),
           "지역C(계절 강함)": synth_region(3, 0.20, 5.0, 0.3), "전국(제공 자료)": y.astype(float)}
rows = []
for name, s in regions.items():
    for kind in ("ridge", "hgb"):
        R_ = rolling_origin(s, n_folds=6, fold=24, kind=kind, lags=(1, 2, 3, 12))
        rows.append({"region": name, "model": kind, "mean_RMSE": R_["RMSE"].mean(), "mean_RMSE_naive": R_["RMSE_naive"].mean(),
                     "상대(RMSE/naive)": R_["RMSE"].mean() / R_["RMSE_naive"].mean(), "worst_fold_상대": (R_["RMSE"] / R_["RMSE_naive"]).max()})
tab = pd.DataFrame(rows).set_index(["region", "model"]); show(tab)
print("\n해석: 상대 RMSE<1이면 나이브보다 낫다. 지역에 따라 순위/이득이 바뀌는지, worst_fold에서 무너지는지 본다.")

## 3. 설정 민감도

In [ ]:
# ---- 하이퍼 설정 민감도: 학습 창 길이(전체 vs 최근 W) ----
s0 = regions["전국(제공 자료)"]; rows = []
for win in (None, 60, 120):
    R_ = rolling_origin(s0, n_folds=6, fold=24, kind="ridge", window=win, lags=(1, 2, 3, 12))
    rows.append({"학습 창": "전체(expanding)" if win is None else f"최근 {win}개월", "mean_RMSE": R_["RMSE"].mean()})
show(pd.DataFrame(rows).set_index("학습 창"))
rows = []
for lg in [(1,), (1, 2, 3), (1, 2, 3, 12), (1, 2, 3, 6, 12, 24)]:
    R_ = rolling_origin(s0, n_folds=6, fold=24, kind="ridge", lags=lg); rows.append({"시차 집합": str(lg), "mean_RMSE": R_["RMSE"].mean()})
show(pd.DataFrame(rows).set_index("시차 집합"))
print("해석: 결론(모형 우열)이 창 길이·시차 선택에 뒤집히면 '강건하지 않다'고 보고한다.")

## 3. 결과 해석 작성
1. 정상 설정과 누수 설정의 성능 차이를 표로 제시하고, **어떤 누수 경로가 가장 큰 왜곡을 만들었는지** 설명한다.
2. 폴드(또는 지역·기간) 간 변동을 보고 "평균은 좋지만 최악의 경우가 나쁘다"를 구분한다.
3. 결론(모형 우열, 효과 방향)이 설정 변경에도 유지되는가? 유지되지 않으면 "강건하다"가 아니라 "민감하다"로 쓴다.
4. 한계: 폴드 수, 이벤트 수, 합성자료의 한계, 위약검정이 모든 식별 가정을 증명하지는 않는다는 점.

**제출:** 정상/누수 비교표, 폴드별 결과, 강건성 표, 한계 단락. 누수 점검 목록(`validation/Leakage_Checklist.md`)과 대조하라.